# Stock NeurIPS2018 Part 1. Data
This series is a reproduction of paper *the process in the paper Practical Deep Reinforcement Learning Approach for Stock Trading*. 

This is the first part of the NeurIPS2018 series, introducing how to use FinRL to fetch and process data that we need for ML/RL trading.

Other demos can be found at the repo of [FinRL-Tutorials]((https://github.com/AI4Finance-Foundation/FinRL-Tutorials)).

# Part 1. Install Packages

In [1]:
import sys
import os

# Get the absolute path of the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the parent directory to sys.path
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [2]:
import pandas as pd
import numpy as np
import datetime
import yfinance as yf

import tinkoff.invest
from tinkoff.invest import CandleInterval

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl import config_tickers
from finrl.config import INDICATORS

import itertools

# Part 2. Fetch data

[yfinance](https://github.com/ranaroussi/yfinance) is an open-source library that provides APIs fetching historical data form Yahoo Finance. In FinRL, we have a class called [YahooDownloader](https://github.com/AI4Finance-Foundation/FinRL/blob/master/finrl/meta/preprocessor/yahoodownloader.py) that use yfinance to fetch data from Yahoo Finance.

**OHLCV**: Data downloaded are in the form of OHLCV, corresponding to **open, high, low, close, volume,** respectively. OHLCV is important because they contain most of numerical information of a stock in time series. From OHLCV, traders can get further judgement and prediction like the momentum, people's interest, market trends, etc.

## Data for a single ticker

Here we provide two ways to fetch data with single ticker, let's take Apple Inc. (AAPL) as an example.

### Using yfinance

In [4]:
aapl_df_yf = yf.download(tickers = "aapl", start='2020-01-01', end='2020-01-31')

[*********************100%***********************]  1 of 1 completed


In [5]:
aapl_df_yf.head()

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2020-01-02,74.059998,75.150002,73.797501,75.087502,73.449394,135480400
2020-01-03,74.287498,75.144997,74.125000,74.357498,72.735321,146322800
2020-01-06,73.447502,74.989998,73.187500,74.949997,73.314873,118387200
2020-01-07,74.959999,75.224998,74.370003,74.597504,72.970093,108872000
2020-01-08,74.290001,76.110001,74.290001,75.797501,74.143898,132079200


### Using FinRL

In FinRL's YahooDownloader, we modified the data frame to the form that convenient for further data processing process. We use adjusted close price instead of close price, and add a column representing the day of a week (0-4 corresponding to Monday-Friday).

In [3]:
raw_df_test = YahooDownloader(start_date = '2024-01-01',
                                end_date = '2020-01-31',
                                ticker_list = ['aapl'])
raw_df_test.head()

NameError: name 'YahooDownloader' is not defined

In [6]:
aapl_df_finrl = YahooDownloader(start_date = '2024-01-01',
                                end_date = '2020-01-31',
                                ticker_list = ['aapl']).fetch_data()

[*********************100%***********************]  1 of 1 completed
Shape of DataFrame:  (20, 8)


In [7]:
aapl_df_finrl.head()

,date,open,high,low,close,volume,tic,day
0,2020-01-02,74.059998,75.150002,73.797501,73.449394,135480400,aapl,3
1,2020-01-03,74.287498,75.144997,74.125000,72.735313,146322800,aapl,4
2,2020-01-06,73.447502,74.989998,73.187500,73.314888,118387200,aapl,0
3,2020-01-07,74.959999,75.224998,74.370003,72.970078,108872000,aapl,1
4,2020-01-08,74.290001,76.110001,74.290001,74.143906,132079200,aapl,2


## Data for the chosen tickers

In [6]:
config_tickers.DOW_30_TICKER

['AXP',
 'AMGN',
 'AAPL',
 'BA',
 'CAT',
 'CSCO',
 'CVX',
 'GS',
 'HD',
 'HON',
 'IBM',
 'INTC',
 'JNJ',
 'KO',
 'JPM',
 'MCD',
 'MMM',
 'MRK',
 'MSFT',
 'NKE',
 'PG',
 'TRV',
 'UNH',
 'CRM',
 'VZ',
 'V',
 'WBA',
 'WMT',
 'DIS',
 'DOW']

In [7]:
TRAIN_START_DATE = pd.Timestamp("2024-12-12T12:00:00Z")
TRAIN_END_DATE = pd.Timestamp("2024-11-12T12:00:00Z")
TRADE_START_DATE = pd.Timestamp("2024-11-12T12:00:00Z")
TRADE_END_DATE = pd.Timestamp("2024-12-12T12:00:00Z")

In [8]:
df_raw = YahooDownloader(start_date = TRAIN_START_DATE,
                     end_date = TRADE_END_DATE,
                     ticker_list = config_tickers.DOW_30_TICKER).fetch_data()































*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (117581, 8)


In [9]:
df_raw.tail()

,date,open,high,low,close,volume,tic,day
117576,2024-11-29,607.868530,610.200012,615.780029,608.799988,2010800,UNH,4
117577,2024-11-29,315.079987,315.079987,316.250000,314.290009,3853300,V,4
117578,2024-11-29,44.340000,44.340000,44.520000,44.380001,9023100,VZ,4
117579,2024-11-29,9.020000,9.020000,9.190000,9.080000,10057000,WBA,4
117580,2024-11-29,92.295509,92.500000,92.639999,92.260002,11041200,WMT,4


## RTS data extraction

In [12]:
from tdata import get_rts_data

TOKEN = 't.DBEaAVcPDDWGkElTk898UNX9yH_ADegVdZgaWKC4a-Ktgz1sbbd-G_9yQf0PeiVEeKuoMWEpUuMT-TNCQzaw1g'

"""
        "CANDLE_INTERVAL_1_MIN": timedelta(days=1),  # Max 1 day for 1-minute candles
        "CANDLE_INTERVAL_5_MIN": timedelta(days=7),  # Max 7 days for 5-minute candles
        "CANDLE_INTERVAL_15_MIN": timedelta(days=14),  # Max 14 days for 15-minute candles
        "CANDLE_INTERVAL_HOUR": timedelta(days=30),  # Max 30 days for hourly candles
        "CANDLE_INTERVAL_DAY": timedelta(days=365),  # Max 1 year for daily candles
"""

if __name__ == "__main__":
    # Set date range for historical data
    end_date = pd.Timestamp("2024-12-31T12:00:00Z")
    start_date = end_date - pd.Timedelta(days=365)
    interval = CandleInterval.CANDLE_INTERVAL_1_MIN
    df = get_rts_data(start_date, end_date, interval, TOKEN)
    df.to_csv("rts_1m_365d.csv", index=False)


ERROR:tinkoff.invest.logging:None GetCandles RESOURCE_EXHAUSTED 


Failed to fetch data for FIGI BBG00475KHX6 (Ticker: TRNFP): (<StatusCode.RESOURCE_EXHAUSTED: (8, 'resource exhausted')>, '', Metadata(tracking_id=None, ratelimit_limit='600, 600;w=60', ratelimit_remaining=0, ratelimit_reset=7, message=None))
Rate limit exceeded. Fetched 7856603 rows, current end is 2024-12-16 12:00:00+00:00 rows left. 
Sleeping for 7 seconds...
All rows fetched


In [16]:
df

,figi,date,open,high,low,close,volume,tic,day
0,BBG004731354,2024-01-02 07:16:00+00:00,598.40,598.40,593.6,593.60,72,ROSN,1
1,BBG004731354,2024-01-02 07:17:00+00:00,598.40,598.55,593.8,598.40,716,ROSN,1
2,BBG004731354,2024-01-02 07:18:00+00:00,598.40,598.55,594.3,598.55,710,ROSN,1
3,BBG004731354,2024-01-02 07:19:00+00:00,598.50,598.55,598.5,598.50,442,ROSN,1
4,BBG004731354,2024-01-02 07:20:00+00:00,598.55,598.55,598.5,598.50,1626,ROSN,1
...,...,...,...,...,...,...,...,...,...
11813883,TCS00A0ZZFS9,2024-12-31 11:54:00+00:00,712.90,712.90,712.9,712.90,2,LEAS,1
11813884,TCS00A0ZZFS9,2024-12-31 11:55:00+00:00,712.90,712.90,712.9,712.90,5,LEAS,1
11813885,TCS00A0ZZFS9,2024-12-31 11:56:00+00:00,713.70,713.70,713.7,713.70,20,LEAS,1
11813886,TCS00A0ZZFS9,2024-12-31 11:57:00+00:00,713.70,713.70,713.7,713.70,2,LEAS,1


# Part 3: Preprocess Data
We need to check for missing data and do feature engineering to convert the data point into a state.
* **Adding technical indicators**. In practical trading, various information needs to be taken into account, such as historical prices, current holding shares, technical indicators, etc. Here, we demonstrate two trend-following technical indicators: MACD and RSI.
* **Adding turbulence index**. Risk-aversion reflects whether an investor prefers to protect the capital. It also influences one's trading strategy when facing different market volatility level. To control the risk in a worst-case scenario, such as financial crisis of 2007–2008, FinRL employs the turbulence index that measures extreme fluctuation of asset price.

Hear let's take **MACD** as an example. Moving average convergence/divergence (MACD) is one of the most commonly used indicator showing bull and bear market. Its calculation is based on EMA (Exponential Moving Average indicator, measuring trend direction over a period of time.)

In [3]:
df_raw = pd.read_csv("rts_15m_90d.csv")

fe = FeatureEngineer(use_technical_indicator=True,
                     tech_indicator_list = INDICATORS,
                     use_vix=True,
                     use_turbulence=True,
                     user_defined_feature = False)

processed = fe.preprocess_data(df_raw)

Successfully added technical indicators
Successfully added turbulence index


In [5]:
list_ticker = processed["tic"].unique().tolist()
list_date = list(pd.date_range(processed['date'].min(),processed['date'].max()).astype(str))
combination = list(itertools.product(list_date,list_ticker))

processed_full = pd.DataFrame(combination,columns=["date","tic"]).merge(processed,on=["date","tic"],how="left")
processed_full = processed_full[processed_full['date'].isin(processed['date'])]
processed_full = processed_full.sort_values(['date','tic'])

processed_full = processed_full.fillna(0)

In [12]:
processed_full.tail()

,date,tic,figi,open,high,low,close,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,turbulence
535,2024-12-11 12:00:00+00:00,OZON,BBG00Y91R9T3,2795.50,2803.00,2791.00,2798.00,33691,2,5.595033,2798.924989,2749.325011,56.554828,200.563948,16.195517,2770.533333,2777.216667,3.619602
536,2024-12-11 12:00:00+00:00,ROSN,BBG004731354,482.30,483.00,482.25,483.00,34804,2,-0.240834,485.269705,481.300295,45.586429,-55.572244,10.161276,483.328333,483.954167,3.619602
537,2024-12-11 12:00:00+00:00,SBER,BBG004730N88,231.18,231.89,231.05,231.56,43304,2,0.065421,232.098843,229.369157,50.927169,147.276067,11.056929,230.701000,230.955167,3.619602
538,2024-12-11 12:00:00+00:00,T,TCS80A107UL4,2343.00,2345.00,2341.80,2343.40,101680,2,-0.588421,2346.000502,2331.859498,47.748787,55.061082,25.641316,2340.653333,2344.816667,3.619602
539,2024-12-11 12:00:00+00:00,VTBR,BBG004730ZJ9,67.14,67.45,67.13,67.28,334640,2,0.025028,67.601546,66.746454,51.755431,80.569176,8.222632,67.115667,67.089667,3.619602


# Part 4: Save the Data

### Split the data for training and trading

In [11]:
def train_test_split(processed_df):
    """
    Splits the processed_df DataFrame into training and testing sets based on calculated date ranges.
    
    :param processed_df: (pd.DataFrame) DataFrame with a date column
    :return: (tuple) Training and testing DataFrames (train_df, test_df)
    """
    # Calculate date ranges using split_dates
    date_ranges = split_dates(processed_full)
    
    # Extract date ranges
    train_start_date = date_ranges["TRAIN_START_DATE"]
    train_end_date = date_ranges["TRAIN_END_DATE"]
    trade_start_date = date_ranges["TRADE_START_DATE"]
    trade_end_date = date_ranges["TRADE_END_DATE"]
    
    # Perform train-test split
    train_df = processed_full[
        (processed_full["date"] >= train_start_date) & 
        (processed_full["date"] <= train_end_date)
    ]
    test_df = processed_full[
        (processed_full["date"] >= trade_start_date) & 
        (processed_full["date"] <= trade_end_date)
    ]
    
    return train_df, test_df

train, trade = train_test_split(processed_full)

print(len(train))
print(len(trade))

TypeError: Cannot convert input [0      2024-09-13 12:00:00+00:00
1      2024-09-13 12:00:00+00:00
2      2024-09-13 12:00:00+00:00
3      2024-09-13 12:00:00+00:00
4      2024-09-13 12:00:00+00:00
                 ...            
535    2024-12-11 12:00:00+00:00
536    2024-12-11 12:00:00+00:00
537    2024-12-11 12:00:00+00:00
538    2024-12-11 12:00:00+00:00
539    2024-12-11 12:00:00+00:00
Name: date, Length: 540, dtype: object] of type <class 'pandas.core.series.Series'> to Timestamp

### Save data to csv file

For Colab users, you can open the virtual directory in colab and manually download the files.

For users running on your local environment, the csv files should be at the same directory of this notebook.

In [13]:
train.to_csv('train_data.csv')
trade.to_csv('trade_data.csv')